## What This Code Does

This notebook transforms raw email texts into **57 numerical features** for spam detection. The feature set consists of:

- **41 word‑frequency features** – normalized counts of common spam‑related words.
- **7 numeric‑pattern features** – detection of specific number patterns (e.g., long digit sequences, area codes).
- **6 symbol‑frequency features** – normalized counts of special characters (`;`, `(`, `[`, `!`, `$`, `#`).
- **3 capitalization features** – statistics on uppercase letter usage.

The script reads a CSV file containing raw email text, applies the feature extraction pipeline, and saves the resulting 57‑dimensional feature matrix to `email_features_57.csv`, ready for classification model training.

In [1]:
import pandas as pd
import re
import numpy as np

# -----------------------------------------------------------------------------
# 57 FEATURES DEFINITION
# -----------------------------------------------------------------------------

# Common word features (full-word match)
KEYWORDS_COUNT = [
    "make", "address", "all", "our", "over", "remove", "internet",
    "order", "mail", "receive", "will", "people", "report", "addresses",
    "free", "business", "email", "you", "credit", "your", "font",
    "money", "hp", "hpl", "george", "lab", "labs", "telnet",
    "data", "technology", "parts", "pm", "direct",
    "cs", "meeting", "original", "project", "re", "edu", "table", "conference"
]

# Numeric pattern features
NUM_PATTERNS = {
    "num3d":   lambda w: w.isdigit() and len(w) >= 3,          # 3+ consecutive digits
    "num000":  lambda w: w.isdigit() and w.endswith("000"),    # ends with '000'
    "num650":  lambda w: w == "650",                           # area code 650
    "num857":  lambda w: w == "857",                           # area code 857
    "num415":  lambda w: w == "415",                           # area code 415
    "num85":   lambda w: w == "85",                            # number 85
    "num1999": lambda w: w.isdigit() and len(w) == 4,          # 4-digit number (year pattern)
}

# Symbol features
SYMBOL_MAP = {
    "charSemicolon":     ";",
    "charRoundbracket":  "(",
    "charSquarebracket": "[",
    "charExclamation":   "!",
    "charDollar":        "$",
    "charHash":          "#"
}

# Capital letter features (raw statistics)
CAPITAL_FEATURES = ["capitalAve", "capitalLong", "capitalTotal"]

# -----------------------------------------------------------------------------
# TEXT PROCESSING FUNCTIONS
# -----------------------------------------------------------------------------

def tokenize(text):
    """
    Extract lowercase alphanumeric tokens (pure numbers are also treated as words).
    Uses [a-z0-9]+ to avoid underscores.
    """
    text = str(text).lower()
    return re.findall(r'[a-z0-9]+', text)

def count_keyword(words, keyword):
    """Count full-word matches for a given keyword (case-insensitive)."""
    return words.count(keyword.lower())

def count_num_pattern(words, pattern_func):
    """Count how many tokens satisfy a numeric pattern function."""
    return sum(1 for w in words if pattern_func(w))

def count_symbol(text, symbol):
    """Count occurrences of a specific symbol."""
    return str(text).count(symbol)

def calc_capital_features(text):
    """Calculate capitalization statistics: average run length, longest run, total uppercase letters."""
    text = str(text)
    current_run = 0
    runs = []
    total = 0
    for c in text:
        if c.isupper():
            current_run += 1
            total += 1
        else:
            if current_run > 0:
                runs.append(current_run)
                current_run = 0
    if current_run > 0:
        runs.append(current_run)

    long_run = max(runs) if runs else 0
    avg_run = np.mean(runs) if runs else 0.0
    return avg_run, long_run, total

# -----------------------------------------------------------------------------
# FEATURE EXTRACTION MAIN FUNCTION
# -----------------------------------------------------------------------------

def extract_features(df):
    # Pre-tokenize once to avoid repeated work
    words_series = df["text"].apply(tokenize)
    total_words = words_series.apply(len)
    total_chars = df["text"].apply(lambda x: len(str(x)))

    # 1. Word frequency features: occurrences / total words
    for kw in KEYWORDS_COUNT:
        count = words_series.apply(lambda words: count_keyword(words, kw))
        df[kw] = np.where(total_words > 0, count / total_words, 0.0)

    # 2. Numeric pattern features: matches / total words
    for num_feat, pattern_func in NUM_PATTERNS.items():
        match_count = words_series.apply(lambda words: count_num_pattern(words, pattern_func))
        df[num_feat] = np.where(total_words > 0, match_count / total_words, 0.0)

    # 3. Symbol frequency features: occurrences / total characters
    for col_name, symbol in SYMBOL_MAP.items():
        count = df["text"].apply(lambda x: count_symbol(x, symbol))
        df[col_name] = np.where(total_chars > 0, count / total_chars, 0.0)

    # 4. Capitalization features (absolute counts)
    cap_values = df["text"].apply(calc_capital_features)
    df["capitalAve"]   = [v[0] for v in cap_values]
    df["capitalLong"]  = [v[1] for v in cap_values]
    df["capitalTotal"] = [v[2] for v in cap_values]

    return df

# -----------------------------------------------------------------------------
# MAIN PIPELINE
# -----------------------------------------------------------------------------
if __name__ == "__main__":
    df = pd.read_csv("combined_data.csv", encoding="utf-8-sig")

    # Optional: remove completely blank rows (uncomment if needed)
    # df = df[df["text"].apply(lambda x: len(str(x).strip()) > 0)].copy()

    df = extract_features(df)

    # Save final feature file (new file name)
    output_filename = "email_features_57.csv"
    df.to_csv(output_filename, index=False, encoding="utf-8-sig")

    # Output summary
    total_feat = len(KEYWORDS_COUNT) + len(NUM_PATTERNS) + len(SYMBOL_MAP) + len(CAPITAL_FEATURES)
    print("Feature extraction completed.")
    print(f"Total features generated: {total_feat}")
    print(f"  - Common words: {len(KEYWORDS_COUNT)}")
    print(f"  - Numeric patterns: {len(NUM_PATTERNS)}")
    print(f"  - Symbols: {len(SYMBOL_MAP)}")
    print(f"  - Capitalization: {len(CAPITAL_FEATURES)}")
    print(f"Output file: {output_filename}")

    # Diagnostic check for the last 3 rows (to spot empty texts)
    print("\n---- Last 3 rows diagnostic ----")
    for i, idx in enumerate(df.index[-3:]):
        text_sample = df.at[idx, "text"] if "text" in df.columns else "[deleted]"
        tokens = tokenize(text_sample)
        print(f"Row {idx} (original text length={len(str(text_sample))}):")
        print(f"  Text preview: {str(text_sample)[:100]}...")
        print(f"  Tokens ({len(tokens)} words): {tokens[:10]}{'...' if len(tokens)>10 else ''}")
        # Check a few feature values
        vals = {feat: df.at[idx, feat] for feat in ["make", "num3d", "charSemicolon", "capitalTotal"]}
        print(f"  Sample feature values: {vals}\n")

Feature extraction completed.
Total features generated: 57
  - Common words: 41
  - Numeric patterns: 7
  - Symbols: 6
  - Capitalization: 3
Output file: email_features_57.csv

---- Last 3 rows diagnostic ----
Row 83445 (original text length=534):
  Text preview: dear valued member canadianpharmacy provides a wide range of pharmaceutical products you will be sur...
  Tokens (76 words): ['dear', 'valued', 'member', 'canadianpharmacy', 'provides', 'a', 'wide', 'range', 'of', 'pharmaceutical']...
  Sample feature values: {'make': np.float64(0.0), 'num3d': np.float64(0.0), 'charSemicolon': np.float64(0.0), 'capitalTotal': np.int64(0)}

Row 83446 (original text length=2113):
  Text preview: subscribe change profile contact us long term escapenumber day trend weather maps waterloo on monday...
  Tokens (297 words): ['subscribe', 'change', 'profile', 'contact', 'us', 'long', 'term', 'escapenumber', 'day', 'trend']...
  Sample feature values: {'make': np.float64(0.0), 'num3d': np.float64(0.0),

## Single-Model Baseline Evaluation

Train and evaluate eight classical classifiers using default hyperparameters. The results serve as a performance baseline for the cascade ensemble.

In [2]:
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

plt.rcParams["font.family"] = ["DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

trained_models = {}

def load_dataset(file_path="email_features_57.csv"):
    """Load the feature-engineered dataset."""
    try:
        df = pd.read_csv(file_path, encoding="utf-8-sig")
    except FileNotFoundError:
        raise FileNotFoundError(
            f"File '{file_path}' not found. Please run feature extraction first "
            "to generate the feature file (e.g., email_features_57.csv)."
        )

    # Rename old label column if necessary
    if "type" in df.columns and "label" not in df.columns:
        df.rename(columns={"type": "label"}, inplace=True)

    # Drop the original text column if present
    if "text" in df.columns:
        df.drop(columns=["text"], inplace=True)

    # Ensure label column exists
    if "label" not in df.columns:
        raise KeyError(
            "The CSV file must contain a 'label' or 'type' column. "
            "Please check your data."
        )

    # Remove rows where all feature values are zero (likely empty texts)
    feature_cols = [c for c in df.columns if c != "label"]
    zero_mask = df[feature_cols].sum(axis=1) == 0
    removed = zero_mask.sum()
    if removed > 0:
        print(f"Removed {removed} all‑zero feature row(s) (probable empty text).")
        df = df[~zero_mask].reset_index(drop=True)

    X = df.drop("label", axis=1)
    y = df["label"]
    print(f"Dataset loaded: {len(df)} records")
    print(f"Feature dimensions: {X.shape[1]} features")
    return X, y

def split_dataset(X, y, test_size=0.2, random_state=42):
    """Stratified train-test split."""
    return train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=y)

def train_all_models(X_train, X_test, y_train, y_test):
    """Train all baseline models and collect performance metrics."""
    models = {
        "Naive_Bayes": MultinomialNB(),
        "Logistic_Regression": LogisticRegression(max_iter=1000, random_state=42),
        "LinearSVC": CalibratedClassifierCV(LinearSVC(max_iter=10000, random_state=42)),
        "Decision_Tree": DecisionTreeClassifier(random_state=42),
        "KNN": KNeighborsClassifier(),
        "Random_Forest": RandomForestClassifier(random_state=42),
        "XGBoost": XGBClassifier(random_state=42, eval_metric="logloss"),
        "LightGBM": LGBMClassifier(random_state=42, verbose=-1),
    }

    results = []
    for name, model in models.items():
        start = time.time()
        model.fit(X_train, y_train)
        trained_models[name] = model
        y_pred = model.predict(X_test)

        train_time = round(time.time() - start, 4)
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        results.append([name, acc, prec, rec, f1, train_time])

    result_df = pd.DataFrame(results, columns=["Model", "Accuracy", "Precision", "Recall", "F1_Score", "Train_Time"])
    return result_df.sort_values(by="F1_Score", ascending=False)

def plot_model_comparison(result_df):
    """Plot comparison bar charts for accuracy, precision, recall, and F1."""
    metrics = ["Accuracy", "Precision", "Recall", "F1_Score"]
    plt.figure(figsize=(14, 8))
    for i, metric in enumerate(metrics, 1):
        plt.subplot(2, 2, i)
        sns.barplot(x=metric, y="Model", data=result_df)
        plt.title(f"Model Comparison - {metric}")
        plt.tight_layout()
    plt.savefig("model_comparison.png", dpi=300, bbox_inches="tight")
    plt.close()
    print("Performance plot saved as: model_comparison.png")

if __name__ == "__main__":
    print("=== Spam Email Classification Pipeline ===")
    X, y = load_dataset()
    X_train, X_test, y_train, y_test = split_dataset(X, y)

    print("\nTraining all classification models...")
    results = train_all_models(X_train, X_test, y_train, y_test)

    print("\n=== Model Performance Ranking (by F1 Score) ===")
    print(results.round(4))

    results.to_csv("model_performance.csv", index=False, encoding="utf-8-sig")
    print("\nResults saved to: model_performance.csv")

    plot_model_comparison(results)
    print("\nPipeline completed successfully.")

=== Spam Email Classification Pipeline ===
Removed 7164 all‑zero feature row(s) (probable empty text).
Dataset loaded: 76284 records
Feature dimensions: 57 features

Training all classification models...

=== Model Performance Ranking (by F1 Score) ===
                 Model  Accuracy  Precision  Recall  F1_Score  Train_Time
5        Random_Forest    0.9021     0.8987  0.9109    0.9047     14.4640
6              XGBoost    0.8884     0.8832  0.9005    0.8918      0.6433
7             LightGBM    0.8813     0.8690  0.9037    0.8860      0.5551
4                  KNN    0.8640     0.8579  0.8792    0.8684      2.4164
3        Decision_Tree    0.8612     0.8620  0.8670    0.8645      1.3284
2            LinearSVC    0.7743     0.8025  0.7400    0.7700      0.9941
1  Logistic_Regression    0.7370     0.7815  0.6728    0.7231      0.2095
0          Naive_Bayes    0.6143     0.5738  0.9502    0.7155      0.0318

Results saved to: model_performance.csv
Performance plot saved as: model_compari

## Tuned Single‑Model Baseline

Load the 57‑dimensional features, apply scaling (MinMax for MNB, Standard for the others), train eight classifiers with tuned hyperparameters, and evaluate accuracy, precision, recall, F1‑score, and runtime.  
Results are saved to `baseline_model_results.csv`.

In [4]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

# =========================================================
# Load Dataset
# =========================================================
print("Loading dataset...")

# Changed to the latest feature file name
df = pd.read_csv("email_features_57.csv", encoding="utf-8-sig")

# Remove rows with missing labels
df = df.dropna(subset=["label"])

# Drop the original text column if it still exists
if "text" in df.columns:
    df.drop(columns=["text"], inplace=True)

# Features and labels
X = df.drop("label", axis=1)
y = df["label"]

print(f"Dataset Size: {len(df)}")
print(f"Feature Count: {X.shape[1]}")

# =========================================================
# Train Test Split
# =========================================================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# =========================================================
# Feature Scaling
# =========================================================

# StandardScaler (for most models)
scaler_standard = StandardScaler()
X_train_standard = scaler_standard.fit_transform(X_train)
X_test_standard = scaler_standard.transform(X_test)

# MinMaxScaler (used exclusively for Multinomial Naive Bayes)
scaler_minmax = MinMaxScaler()
X_train_minmax = scaler_minmax.fit_transform(X_train)
X_test_minmax = scaler_minmax.transform(X_test)

# =========================================================
# Model Definitions (tuned for the spam dataset)
# =========================================================
model_defs = {

    "MNB": MultinomialNB(alpha=0.1),

    "LR": LogisticRegression(
        max_iter=5000,
        random_state=42,
        C=1.0,
        class_weight="balanced"
    ),

    "SVM": CalibratedClassifierCV(
        LinearSVC(
            max_iter=20000,
            random_state=42,
            C=0.5,
            class_weight="balanced"
        )
    ),

    "DT": DecisionTreeClassifier(
        max_depth=12,
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=42,
        class_weight="balanced"
    ),

    "KNN": KNeighborsClassifier(
        n_neighbors=7,
        weights="distance"
    ),

    "RF": RandomForestClassifier(
        n_estimators=150,
        max_depth=12,
        min_samples_split=10,
        n_jobs=-1,
        random_state=42,
        class_weight="balanced"
    ),

    "LGBM": LGBMClassifier(
        n_estimators=200,
        max_depth=8,
        learning_rate=0.05,
        n_jobs=-1,
        random_state=42,
        verbose=-1
    ),

    "XGB": XGBClassifier(
        n_estimators=200,
        max_depth=8,
        learning_rate=0.05,
        n_jobs=-1,
        eval_metric="logloss",
        random_state=42
    )
}

# =========================================================
# Model Training + Evaluation
# =========================================================
trained_models = {}
results = []

print("\nStarting model training...\n")

for name, model in model_defs.items():
    print(f"Training {name} ...")

    # Data scaling rule:
    #   MNB  -> MinMax (requires non‑negative features)
    #   All other models -> Standard scaler
    if name == "MNB":
        X_train_used = X_train_minmax
        X_test_used = X_test_minmax
    else:
        X_train_used = X_train_standard
        X_test_used = X_test_standard

    # Training
    start_time = time.time()
    model.fit(X_train_used, y_train)

    # Prediction
    y_pred = model.predict(X_test_used)
    elapsed_time = round(time.time() - start_time, 4)

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    # Save model and results
    trained_models[name] = model
    results.append([name, accuracy, precision, recall, f1, elapsed_time])

    print(f"{name} completed | F1: {f1:.4f} | Time: {elapsed_time}s")

# =========================================================
# Results Output
# =========================================================
results_df = pd.DataFrame(
    results,
    columns=["Model", "Accuracy", "Precision", "Recall", "F1_Score", "Runtime"]
)
results_df = results_df.sort_values(by="F1_Score", ascending=False)

results_df.to_csv("baseline_model_results.csv", index=False, encoding="utf-8-sig")

print("\n==============================")
print("Baseline Model Results")
print("==============================")
print(results_df.round(4))

print("\nResults saved to: baseline_model_results.csv")
print("\nAll baseline models trained successfully.")

Loading dataset...
Dataset Size: 83448
Feature Count: 57

Starting model training...

Training MNB ...
MNB completed | F1: 0.7841 | Time: 0.0243s
Training LR ...
LR completed | F1: 0.7732 | Time: 0.175s
Training SVM ...
SVM completed | F1: 0.8191 | Time: 9.8543s
Training DT ...
DT completed | F1: 0.8543 | Time: 0.6238s
Training KNN ...
KNN completed | F1: 0.8720 | Time: 0.9166s
Training RF ...
RF completed | F1: 0.8694 | Time: 1.1167s
Training LGBM ...


C:\conda\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LGBM completed | F1: 0.8775 | Time: 0.9077s
Training XGB ...
XGB completed | F1: 0.8821 | Time: 1.4431s

Baseline Model Results
  Model  Accuracy  Precision  Recall  F1_Score  Runtime
7   XGB    0.8700     0.8439  0.9239    0.8821   1.4431
6  LGBM    0.8650     0.8397  0.9189    0.8775   0.9077
4   KNN    0.8596     0.8378  0.9090    0.8720   0.9166
5    RF    0.8545     0.8238  0.9204    0.8694   1.1167
3    DT    0.8357     0.8006  0.9157    0.8543   0.6238
2   SVM    0.7936     0.7601  0.8880    0.8191   9.8543
0   MNB    0.7223     0.6636  0.9580    0.7841   0.0243
1    LR    0.7642     0.7827  0.7639    0.7732   0.1750

Results saved to: baseline_model_results.csv

All baseline models trained successfully.


## Random Cascade Order Search (100 Permutations)

Train the eight tuned classifiers, evaluate 100 unique random cascade orders with a confidence threshold of 0.90, compute the composite score \(y(\pi)\), and display the top‑10 sequences and the overall best order.

In [6]:
import pandas as pd
import numpy as np
import time
import random
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

# =========================================================
# 1. Data loading and preprocessing
# =========================================================
print("Loading dataset...")
df = pd.read_csv("email_features_57.csv", encoding="utf-8-sig")

# Rename old 'type' label column if necessary
if "type" in df.columns and "label" not in df.columns:
    df.rename(columns={"type": "label"}, inplace=True)

# Remove rows with missing labels
df = df.dropna(subset=["label"])

# Drop the original text column if it still exists
if "text" in df.columns:
    df.drop(columns=["text"], inplace=True)

# Features and labels
X = df.drop("label", axis=1)
y = df["label"].astype(int)   # Ensure labels are integers 0/1

# Remove rows where all feature values are zero (empty texts)
feature_cols = X.columns
zero_mask = (X[feature_cols].sum(axis=1) == 0)
if zero_mask.sum() > 0:
    print(f"Removed {zero_mask.sum()} all‑zero feature rows")
    X = X[~zero_mask]
    y = y[~zero_mask]

print(f"Dataset size: {len(X)}")
print(f"Feature count: {X.shape[1]}")

# =========================================================
# 2. Train–test split
# =========================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# =========================================================
# 3. Feature scaling (MNB uses MinMax, others use Standard)
# =========================================================
scaler_standard = StandardScaler()
X_train_standard = scaler_standard.fit_transform(X_train)
X_test_standard = scaler_standard.transform(X_test)

scaler_minmax = MinMaxScaler()
X_train_minmax = scaler_minmax.fit_transform(X_train)
X_test_minmax = scaler_minmax.transform(X_test)

# Dictionary for quick access during cascade evaluation
X_test_dict = {
    "minmax": X_test_minmax,
    "standard": X_test_standard
}

# =========================================================
# 4. Model definitions (tuned hyperparameters)
# =========================================================
model_defs = {
    "MNB": MultinomialNB(alpha=0.1),

    "LR": LogisticRegression(
        max_iter=5000, random_state=42, C=1.0, class_weight="balanced"
    ),
    "SVM": CalibratedClassifierCV(
        LinearSVC(max_iter=20000, random_state=42, C=0.5, class_weight="balanced")
    ),
    "DT": DecisionTreeClassifier(
        max_depth=12, min_samples_split=10, min_samples_leaf=5,
        random_state=42, class_weight="balanced"
    ),
    "KNN": KNeighborsClassifier(n_neighbors=7, weights="distance"),
    "RF": RandomForestClassifier(
        n_estimators=150, max_depth=12, min_samples_split=10,
        n_jobs=-1, random_state=42, class_weight="balanced"
    ),
    "LGBM": LGBMClassifier(
        n_estimators=200, max_depth=8, learning_rate=0.05,
        n_jobs=-1, random_state=42, verbose=-1
    ),
    "XGB": XGBClassifier(
        n_estimators=200, max_depth=8, learning_rate=0.05,
        n_jobs=-1, eval_metric="logloss", random_state=42
    )
}

# =========================================================
# 5. Train all models and save them
# =========================================================
trained_models = {}
results_baseline = []

print("\n=== Starting model training ===\n")
for name, model in model_defs.items():
    print(f"Training {name} ...")

    # Select appropriately scaled data
    if name == "MNB":
        X_tr = X_train_minmax
        X_te = X_test_minmax
    else:
        X_tr = X_train_standard
        X_te = X_test_standard

    start = time.time()
    model.fit(X_tr, y_train)
    train_time = round(time.time() - start, 4)

    y_pred = model.predict(X_te)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    trained_models[name] = model
    results_baseline.append([name, acc, prec, rec, f1, train_time])
    print(f"{name} done | F1: {f1:.4f} | Time: {train_time}s")

# Save standalone model performance
baseline_df = pd.DataFrame(
    results_baseline,
    columns=["Model", "Accuracy", "Precision", "Recall", "F1_Score", "Runtime"]
).sort_values("F1_Score", ascending=False)
baseline_df.to_csv("baseline_model_results.csv", index=False, encoding="utf-8-sig")
print("\nBaseline results saved to baseline_model_results.csv")

# =========================================================
# 6. Random cascade order search (100 unique permutations)
# =========================================================
print("\n=== Generating 100 random cascade orders ===")
base_models = list(model_defs.keys())
random.seed(42)
random_orders = []
while len(random_orders) < 100:
    order = random.sample(base_models, len(base_models))
    if order not in random_orders:
        random_orders.append(order)
print("Generated 100 unique random orders.\n")

cascade_results = []
CONF_THRESHOLD = 0.90   # Confidence threshold for early decision

for idx, order in enumerate(random_orders, 1):
    start_time = time.time()

    # Final prediction container
    final_pred = np.full(len(y_test), -1)
    unresolved_idx = np.arange(len(y_test))

    # Cascade prediction
    for model_name in order:
        if len(unresolved_idx) == 0:
            break

        model = trained_models[model_name]

        # Select correctly scaled data
        if model_name == "MNB":
            X_current = X_test_dict["minmax"][unresolved_idx]
        else:
            X_current = X_test_dict["standard"][unresolved_idx]

        probas = model.predict_proba(X_current)
        preds = np.argmax(probas, axis=1)
        confidence = np.max(probas, axis=1)

        # Accept high‑confidence predictions
        resolved_mask = confidence >= CONF_THRESHOLD
        resolved_indices = unresolved_idx[resolved_mask]
        final_pred[resolved_indices] = preds[resolved_mask]

        # Keep unresolved samples for the next model
        unresolved_idx = unresolved_idx[~resolved_mask]

    # Fallback: use the last model's hard predictions for any remaining samples
    if len(unresolved_idx) > 0:
        fallback_model_name = order[-1]
        fallback_model = trained_models[fallback_model_name]

        if fallback_model_name == "MNB":
            X_fb = X_test_dict["minmax"][unresolved_idx]
        else:
            X_fb = X_test_dict["standard"][unresolved_idx]

        fallback_preds = fallback_model.predict(X_fb)
        final_pred[unresolved_idx] = fallback_preds

    elapsed = round(time.time() - start_time, 4)
    acc = accuracy_score(y_test, final_pred)
    f1 = f1_score(y_test, final_pred, zero_division=0)

    cascade_results.append({
        "order_id": idx,
        "model_sequence": " -> ".join(order),
        "time_seconds": elapsed,
        "accuracy": round(acc, 4),
        "f1_score": round(f1, 4)
    })

# Convert to DataFrame
cascade_df = pd.DataFrame(cascade_results)

# Speed score (Min–Max normalization)
T_min = cascade_df["time_seconds"].min()
T_max = cascade_df["time_seconds"].max()
if T_max == T_min:
    cascade_df["speed_score"] = 1.0
else:
    cascade_df["speed_score"] = 1 - (cascade_df["time_seconds"] - T_min) / (T_max - T_min)

# Composite score: 85% F1, 15% speed
cascade_df["composite_score"] = (
    0.85 * cascade_df["f1_score"] + 0.15 * cascade_df["speed_score"]
)

# Sort by composite score descending
cascade_df = cascade_df.sort_values("composite_score", ascending=False)

# Save results
cascade_df.round(4).to_csv("random_cascade_results.csv", index=False, encoding="utf-8-sig")

print("\nTop 10 cascade orders:")
print(cascade_df.head(10).to_string(index=False))
print("\nResults saved to: random_cascade_results.csv")
print("Pipeline completed successfully.")
best_order = results_df.iloc[0]

print("\n" + "="*80)
print("          🏆 BEST CASCADE ORDER (HIGHEST COMPOSITE SCORE) 🏆")
print("="*80)
print(f"Model Sequence:   {best_order['model_sequence']}")
print(f"Composite Score:   {best_order['composite_score']}")
print(f"F1 Score:          {best_order['f1_score']}")
print(f"Accuracy:          {best_order['accuracy']}")
print(f"Time Seconds:      {best_order['time_seconds']}")
print("="*80 + "\n")

Loading dataset...
Removed 7164 all‑zero feature rows
Dataset size: 76284
Feature count: 57

=== Starting model training ===

Training MNB ...
MNB done | F1: 0.7909 | Time: 0.0168s
Training LR ...
LR done | F1: 0.8158 | Time: 0.166s
Training SVM ...
SVM done | F1: 0.8101 | Time: 6.7665s
Training DT ...
DT done | F1: 0.8563 | Time: 0.6915s
Training KNN ...
KNN done | F1: 0.8768 | Time: 0.0186s
Training RF ...
RF done | F1: 0.8666 | Time: 1.0393s
Training LGBM ...


C:\conda\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LGBM done | F1: 0.8820 | Time: 0.84s
Training XGB ...
XGB done | F1: 0.8867 | Time: 1.3135s

Baseline results saved to baseline_model_results.csv

=== Generating 100 random cascade orders ===
Generated 100 unique random orders.



C:\conda\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\conda\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\conda\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\conda\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\conda\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\conda\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but 


Top 10 cascade orders:
 order_id                                     model_sequence  time_seconds  accuracy  f1_score  speed_score  composite_score
       51 DT -> XGB -> SVM -> RF -> MNB -> LR -> KNN -> LGBM        0.3559    0.8831    0.8875     0.973122         0.900343
       10 DT -> LR -> XGB -> RF -> SVM -> KNN -> MNB -> LGBM        0.3595    0.8828    0.8872     0.968704         0.899426
       32 MNB -> DT -> SVM -> LGBM -> RF -> LR -> KNN -> XGB        0.3667    0.8846    0.8881     0.959867         0.898865
       58 MNB -> SVM -> LGBM -> LR -> DT -> RF -> KNN -> XGB        0.3642    0.8837    0.8870     0.962936         0.898390
       49 SVM -> LGBM -> RF -> MNB -> DT -> KNN -> LR -> XGB        0.3801    0.8839    0.8872     0.943422         0.895633
       50 DT -> LGBM -> XGB -> SVM -> KNN -> MNB -> RF -> LR        0.3340    0.8702    0.8741     1.000000         0.892985
       54 DT -> SVM -> MNB -> XGB -> RF -> LGBM -> KNN -> LR        0.3364    0.8700    0.8739     0.

C:\conda\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\conda\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## Random Cascade Order Search (1000 Permutations)

In [10]:
import pandas as pd
import numpy as np
import time
import random
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

# =========================================================
# 1. Data loading and preprocessing
# =========================================================
print("Loading dataset...")
df = pd.read_csv("email_features_57.csv", encoding="utf-8-sig")

# Rename old 'type' label column if necessary
if "type" in df.columns and "label" not in df.columns:
    df.rename(columns={"type": "label"}, inplace=True)

# Remove rows with missing labels
df = df.dropna(subset=["label"])

# Drop the original text column if it still exists
if "text" in df.columns:
    df.drop(columns=["text"], inplace=True)

# Features and labels
X = df.drop("label", axis=1)
y = df["label"].astype(int)   # Ensure labels are integers 0/1

# Remove rows where all feature values are zero (empty texts)
feature_cols = X.columns
zero_mask = (X[feature_cols].sum(axis=1) == 0)
if zero_mask.sum() > 0:
    print(f"Removed {zero_mask.sum()} all‑zero feature rows")
    X = X[~zero_mask]
    y = y[~zero_mask]

print(f"Dataset size: {len(X)}")
print(f"Feature count: {X.shape[1]}")

# =========================================================
# 2. Train–test split
# =========================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# =========================================================
# 3. Feature scaling (MNB uses MinMax, others use Standard)
# =========================================================
scaler_standard = StandardScaler()
X_train_standard = scaler_standard.fit_transform(X_train)
X_test_standard = scaler_standard.transform(X_test)

scaler_minmax = MinMaxScaler()
X_train_minmax = scaler_minmax.fit_transform(X_train)
X_test_minmax = scaler_minmax.transform(X_test)

# Dictionary for quick access during cascade evaluation
X_test_dict = {
    "minmax": X_test_minmax,
    "standard": X_test_standard
}

# =========================================================
# 4. Model definitions (tuned hyperparameters)
# =========================================================
model_defs = {
    "MNB": MultinomialNB(alpha=0.1),

    "LR": LogisticRegression(
        max_iter=5000, random_state=42, C=1.0, class_weight="balanced"
    ),
    "SVM": CalibratedClassifierCV(
        LinearSVC(max_iter=20000, random_state=42, C=0.5, class_weight="balanced")
    ),
    "DT": DecisionTreeClassifier(
        max_depth=12, min_samples_split=10, min_samples_leaf=5,
        random_state=42, class_weight="balanced"
    ),
    "KNN": KNeighborsClassifier(n_neighbors=7, weights="distance"),
    "RF": RandomForestClassifier(
        n_estimators=150, max_depth=12, min_samples_split=10,
        n_jobs=-1, random_state=42, class_weight="balanced"
    ),
    "LGBM": LGBMClassifier(
        n_estimators=200, max_depth=8, learning_rate=0.05,
        n_jobs=-1, random_state=42, verbose=-1
    ),
    "XGB": XGBClassifier(
        n_estimators=200, max_depth=8, learning_rate=0.05,
        n_jobs=-1, eval_metric="logloss", random_state=42
    )
}

# =========================================================
# 5. Train all models and save them
# =========================================================
trained_models = {}
results_baseline = []

print("\n=== Starting model training ===\n")
for name, model in model_defs.items():
    print(f"Training {name} ...")

    # Select appropriately scaled data
    if name == "MNB":
        X_tr = X_train_minmax
        X_te = X_test_minmax
    else:
        X_tr = X_train_standard
        X_te = X_test_standard

    start = time.time()
    model.fit(X_tr, y_train)
    train_time = round(time.time() - start, 4)

    y_pred = model.predict(X_te)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    trained_models[name] = model
    results_baseline.append([name, acc, prec, rec, f1, train_time])
    print(f"{name} done | F1: {f1:.4f} | Time: {train_time}s")

# Save standalone model performance
baseline_df = pd.DataFrame(
    results_baseline,
    columns=["Model", "Accuracy", "Precision", "Recall", "F1_Score", "Runtime"]
).sort_values("F1_Score", ascending=False)
baseline_df.to_csv("baseline_model_results.csv", index=False, encoding="utf-8-sig")
print("\nBaseline results saved to baseline_model_results.csv")

# =========================================================
# 6. Random cascade order search (1000 unique permutations)
# =========================================================
print("\n=== Generating 1000 random cascade orders ===")
base_models = list(model_defs.keys())
random.seed(42)
random_orders = []
while len(random_orders) < 1000:
    order = random.sample(base_models, len(base_models))
    if order not in random_orders:
        random_orders.append(order)
print(f"Generated {len(random_orders)} unique random orders.\n")

cascade_results = []
CONF_THRESHOLD = 0.90   # Confidence threshold for early decision

for idx, order in enumerate(random_orders, 1):
    start_time = time.time()

    # Final prediction container
    final_pred = np.full(len(y_test), -1)
    unresolved_idx = np.arange(len(y_test))

    # Cascade prediction
    for model_name in order:
        if len(unresolved_idx) == 0:
            break

        model = trained_models[model_name]

        # Select correctly scaled data
        if model_name == "MNB":
            X_current = X_test_dict["minmax"][unresolved_idx]
        else:
            X_current = X_test_dict["standard"][unresolved_idx]

        probas = model.predict_proba(X_current)
        preds = np.argmax(probas, axis=1)
        confidence = np.max(probas, axis=1)

        # Accept high‑confidence predictions
        resolved_mask = confidence >= CONF_THRESHOLD
        resolved_indices = unresolved_idx[resolved_mask]
        final_pred[resolved_indices] = preds[resolved_mask]

        # Keep unresolved samples for the next model
        unresolved_idx = unresolved_idx[~resolved_mask]

    # Fallback: use the last model's hard predictions for any remaining samples
    if len(unresolved_idx) > 0:
        fallback_model_name = order[-1]
        fallback_model = trained_models[fallback_model_name]

        if fallback_model_name == "MNB":
            X_fb = X_test_dict["minmax"][unresolved_idx]
        else:
            X_fb = X_test_dict["standard"][unresolved_idx]

        fallback_preds = fallback_model.predict(X_fb)
        final_pred[unresolved_idx] = fallback_preds

    elapsed = round(time.time() - start_time, 4)
    acc = accuracy_score(y_test, final_pred)
    f1 = f1_score(y_test, final_pred, zero_division=0)

    cascade_results.append({
        "order_id": idx,
        "model_sequence": " -> ".join(order),
        "time_seconds": elapsed,
        "accuracy": round(acc, 4),
        "f1_score": round(f1, 4)
    })

# Convert to DataFrame and compute scores
cascade_df = pd.DataFrame(cascade_results)

# Speed score (Min–Max normalization)
T_min = cascade_df["time_seconds"].min()
T_max = cascade_df["time_seconds"].max()
if T_max == T_min:
    cascade_df["speed_score"] = 1.0
else:
    cascade_df["speed_score"] = 1 - (cascade_df["time_seconds"] - T_min) / (T_max - T_min)

# Composite score: 85% F1, 15% speed
cascade_df["composite_score"] = (
    0.85 * cascade_df["f1_score"] + 0.15 * cascade_df["speed_score"]
)

# Sort by composite score descending and round values
cascade_df = cascade_df.sort_values("composite_score", ascending=False)
cascade_df = cascade_df.round({
    "time_seconds": 4,
    "accuracy": 4,
    "f1_score": 4,
    "speed_score": 4,
    "composite_score": 4
})

# Save full results
cascade_df.to_csv("1000_random_orders_results.csv", index=False, encoding="utf-8-sig")

# Display top 10
print("\nTop 10 Cascade Orders:")
print(cascade_df.head(10).to_string(index=False))

# Highlight the best order
best_order = cascade_df.iloc[0]
print("\n" + "="*80)
print("          🏆 BEST CASCADE ORDER (HIGHEST COMPOSITE SCORE) 🏆")
print("="*80)
print(f"Model Sequence:   {best_order['model_sequence']}")
print(f"Composite Score:   {best_order['composite_score']}")
print(f"F1 Score:          {best_order['f1_score']}")
print(f"Accuracy:          {best_order['accuracy']}")
print(f"Time Seconds:      {best_order['time_seconds']}")
print("="*80 + "\n")

print("Results saved to: 1000_random_orders_results.csv")
print("Pipeline completed successfully.")

Loading dataset...
Removed 7164 all‑zero feature rows
Dataset size: 76284
Feature count: 57

=== Starting model training ===

Training MNB ...
MNB done | F1: 0.7909 | Time: 0.0186s
Training LR ...
LR done | F1: 0.8158 | Time: 0.1665s
Training SVM ...
SVM done | F1: 0.8101 | Time: 6.6838s
Training DT ...
DT done | F1: 0.8563 | Time: 0.7244s
Training KNN ...
KNN done | F1: 0.8768 | Time: 0.0192s
Training RF ...
RF done | F1: 0.8666 | Time: 1.1198s
Training LGBM ...


C:\conda\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LGBM done | F1: 0.8820 | Time: 0.9892s
Training XGB ...
XGB done | F1: 0.8867 | Time: 1.4952s

Baseline results saved to baseline_model_results.csv

=== Generating 1000 random cascade orders ===
Generated 1000 unique random orders.



C:\conda\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\conda\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\conda\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\conda\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\conda\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\conda\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but 


Top 10 Cascade Orders:
 order_id                                     model_sequence  time_seconds  accuracy  f1_score  speed_score  composite_score
      665 RF -> XGB -> DT -> SVM -> LR -> MNB -> KNN -> LGBM        0.3605    0.8841    0.8885       0.9957           0.9046
      801 DT -> RF -> SVM -> LGBM -> LR -> MNB -> KNN -> XGB        0.3570    0.8845    0.8879       0.9970           0.9043
      454 RF -> DT -> SVM -> LR -> LGBM -> MNB -> KNN -> XGB        0.3596    0.8845    0.8879       0.9960           0.9041
      507 DT -> MNB -> LGBM -> RF -> KNN -> SVM -> LR -> XGB        0.3910    0.8865    0.8899       0.9844           0.9041
      713 DT -> RF -> MNB -> SVM -> LGBM -> KNN -> LR -> XGB        0.3629    0.8846    0.8880       0.9948           0.9040
      687 LGBM -> DT -> MNB -> LR -> RF -> SVM -> KNN -> XGB        0.3796    0.8856    0.8890       0.9886           0.9039
      875 DT -> XGB -> SVM -> KNN -> MNB -> RF -> LR -> LGBM        0.3593    0.8831    0.8875       

## Random Cascade Order Search (5000 Permutations)

In [11]:
import pandas as pd
import numpy as np
import time
import random
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

# =========================================================
# 1. Data loading and preprocessing
# =========================================================
print("Loading dataset...")
df = pd.read_csv("email_features_57.csv", encoding="utf-8-sig")

# Rename old 'type' label column if necessary
if "type" in df.columns and "label" not in df.columns:
    df.rename(columns={"type": "label"}, inplace=True)

# Remove rows with missing labels
df = df.dropna(subset=["label"])

# Drop the original text column if it still exists
if "text" in df.columns:
    df.drop(columns=["text"], inplace=True)

# Features and labels
X = df.drop("label", axis=1)
y = df["label"].astype(int)   # Ensure labels are integers 0/1

# Remove rows where all feature values are zero (empty texts)
feature_cols = X.columns
zero_mask = (X[feature_cols].sum(axis=1) == 0)
if zero_mask.sum() > 0:
    print(f"Removed {zero_mask.sum()} all‑zero feature rows")
    X = X[~zero_mask]
    y = y[~zero_mask]

print(f"Dataset size: {len(X)}")
print(f"Feature count: {X.shape[1]}")

# =========================================================
# 2. Train–test split
# =========================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# =========================================================
# 3. Feature scaling (MNB uses MinMax, others use Standard)
# =========================================================
scaler_standard = StandardScaler()
X_train_standard = scaler_standard.fit_transform(X_train)
X_test_standard = scaler_standard.transform(X_test)

scaler_minmax = MinMaxScaler()
X_train_minmax = scaler_minmax.fit_transform(X_train)
X_test_minmax = scaler_minmax.transform(X_test)

# Dictionary for quick access during cascade evaluation
X_test_dict = {
    "minmax": X_test_minmax,
    "standard": X_test_standard
}

# =========================================================
# 4. Model definitions (tuned hyperparameters)
# =========================================================
model_defs = {
    "MNB": MultinomialNB(alpha=0.1),

    "LR": LogisticRegression(
        max_iter=5000, random_state=42, C=1.0, class_weight="balanced"
    ),
    "SVM": CalibratedClassifierCV(
        LinearSVC(max_iter=20000, random_state=42, C=0.5, class_weight="balanced")
    ),
    "DT": DecisionTreeClassifier(
        max_depth=12, min_samples_split=10, min_samples_leaf=5,
        random_state=42, class_weight="balanced"
    ),
    "KNN": KNeighborsClassifier(n_neighbors=7, weights="distance"),
    "RF": RandomForestClassifier(
        n_estimators=150, max_depth=12, min_samples_split=10,
        n_jobs=-1, random_state=42, class_weight="balanced"
    ),
    "LGBM": LGBMClassifier(
        n_estimators=200, max_depth=8, learning_rate=0.05,
        n_jobs=-1, random_state=42, verbose=-1
    ),
    "XGB": XGBClassifier(
        n_estimators=200, max_depth=8, learning_rate=0.05,
        n_jobs=-1, eval_metric="logloss", random_state=42
    )
}

# =========================================================
# 5. Train all models and save them
# =========================================================
trained_models = {}
results_baseline = []

print("\n=== Starting model training ===\n")
for name, model in model_defs.items():
    print(f"Training {name} ...")

    # Select appropriately scaled data
    if name == "MNB":
        X_tr = X_train_minmax
        X_te = X_test_minmax
    else:
        X_tr = X_train_standard
        X_te = X_test_standard

    start = time.time()
    model.fit(X_tr, y_train)
    train_time = round(time.time() - start, 4)

    y_pred = model.predict(X_te)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    trained_models[name] = model
    results_baseline.append([name, acc, prec, rec, f1, train_time])
    print(f"{name} done | F1: {f1:.4f} | Time: {train_time}s")

# Save standalone model performance
baseline_df = pd.DataFrame(
    results_baseline,
    columns=["Model", "Accuracy", "Precision", "Recall", "F1_Score", "Runtime"]
).sort_values("F1_Score", ascending=False)
baseline_df.to_csv("baseline_model_results.csv", index=False, encoding="utf-8-sig")
print("\nBaseline results saved to baseline_model_results.csv")

# =========================================================
# 6. Random cascade order search (5000 unique permutations)
# =========================================================
print("\n=== Generating 5000 random cascade orders ===")
base_models = list(model_defs.keys())
random.seed(42)
random_orders = []
while len(random_orders) < 5000:
    order = random.sample(base_models, len(base_models))
    if order not in random_orders:
        random_orders.append(order)
print(f"Generated {len(random_orders)} unique random orders.\n")

cascade_results = []
CONF_THRESHOLD = 0.90   # Confidence threshold for early decision

for idx, order in enumerate(random_orders, 1):
    start_time = time.time()

    # Final prediction container
    final_pred = np.full(len(y_test), -1)
    unresolved_idx = np.arange(len(y_test))

    # Cascade prediction
    for model_name in order:
        if len(unresolved_idx) == 0:
            break

        model = trained_models[model_name]

        # Select correctly scaled data
        if model_name == "MNB":
            X_current = X_test_dict["minmax"][unresolved_idx]
        else:
            X_current = X_test_dict["standard"][unresolved_idx]

        probas = model.predict_proba(X_current)
        preds = np.argmax(probas, axis=1)
        confidence = np.max(probas, axis=1)

        # Accept high‑confidence predictions
        resolved_mask = confidence >= CONF_THRESHOLD
        resolved_indices = unresolved_idx[resolved_mask]
        final_pred[resolved_indices] = preds[resolved_mask]

        # Keep unresolved samples for the next model
        unresolved_idx = unresolved_idx[~resolved_mask]

    # Fallback: use the last model's hard predictions for any remaining samples
    if len(unresolved_idx) > 0:
        fallback_model_name = order[-1]
        fallback_model = trained_models[fallback_model_name]

        if fallback_model_name == "MNB":
            X_fb = X_test_dict["minmax"][unresolved_idx]
        else:
            X_fb = X_test_dict["standard"][unresolved_idx]

        fallback_preds = fallback_model.predict(X_fb)
        final_pred[unresolved_idx] = fallback_preds

    elapsed = round(time.time() - start_time, 4)
    acc = accuracy_score(y_test, final_pred)
    f1 = f1_score(y_test, final_pred, zero_division=0)

    cascade_results.append({
        "order_id": idx,
        "model_sequence": " -> ".join(order),
        "time_seconds": elapsed,
        "accuracy": round(acc, 4),
        "f1_score": round(f1, 4)
    })

# Convert to DataFrame and compute scores
cascade_df = pd.DataFrame(cascade_results)

# Speed score (Min–Max normalization)
T_min = cascade_df["time_seconds"].min()
T_max = cascade_df["time_seconds"].max()
if T_max == T_min:
    cascade_df["speed_score"] = 1.0
else:
    cascade_df["speed_score"] = 1 - (cascade_df["time_seconds"] - T_min) / (T_max - T_min)

# Composite score: 85% F1, 15% speed
cascade_df["composite_score"] = (
    0.85 * cascade_df["f1_score"] + 0.15 * cascade_df["speed_score"]
)

# Sort by composite score descending and round values
cascade_df = cascade_df.sort_values("composite_score", ascending=False)
cascade_df = cascade_df.round({
    "time_seconds": 4,
    "accuracy": 4,
    "f1_score": 4,
    "speed_score": 4,
    "composite_score": 4
})

# Save full results
cascade_df.to_csv("5000_random_orders_results.csv", index=False, encoding="utf-8-sig")

# Display top 10
print("\nTop 10 Cascade Orders:")
print(cascade_df.head(10).to_string(index=False))

# Highlight the best order
best_order = cascade_df.iloc[0]
print("\n" + "="*80)
print("          🏆 BEST CASCADE ORDER (HIGHEST COMPOSITE SCORE) 🏆")
print("="*80)
print(f"Model Sequence:   {best_order['model_sequence']}")
print(f"Composite Score:   {best_order['composite_score']}")
print(f"F1 Score:          {best_order['f1_score']}")
print(f"Accuracy:          {best_order['accuracy']}")
print(f"Time Seconds:      {best_order['time_seconds']}")
print("="*80 + "\n")

print("Results saved to: 5000_random_orders_results.csv")
print("Pipeline completed successfully.")

Loading dataset...
Removed 7164 all‑zero feature rows
Dataset size: 76284
Feature count: 57

=== Starting model training ===

Training MNB ...
MNB done | F1: 0.7909 | Time: 0.0178s
Training LR ...
LR done | F1: 0.8158 | Time: 0.3613s
Training SVM ...
SVM done | F1: 0.8101 | Time: 6.8218s
Training DT ...
DT done | F1: 0.8563 | Time: 0.6707s
Training KNN ...
KNN done | F1: 0.8768 | Time: 0.0166s
Training RF ...
RF done | F1: 0.8666 | Time: 1.0417s
Training LGBM ...


C:\conda\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LGBM done | F1: 0.8820 | Time: 1.0222s
Training XGB ...
XGB done | F1: 0.8867 | Time: 1.7111s

Baseline results saved to baseline_model_results.csv

=== Generating 5000 random cascade orders ===
Generated 5000 unique random orders.



C:\conda\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\conda\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\conda\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\conda\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\conda\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\conda\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but 


Top 10 Cascade Orders:
 order_id                                     model_sequence  time_seconds  accuracy  f1_score  speed_score  composite_score
     4685 MNB -> LGBM -> RF -> KNN -> DT -> SVM -> LR -> XGB        0.4982    0.8907    0.8939       0.9999           0.9098
     1256 LGBM -> KNN -> RF -> DT -> LR -> MNB -> SVM -> XGB        0.4930    0.8905    0.8938       0.9999           0.9097
     3713 RF -> LGBM -> KNN -> DT -> MNB -> LR -> SVM -> XGB        0.5129    0.8905    0.8938       0.9999           0.9097
     4510 LGBM -> KNN -> DT -> RF -> LR -> MNB -> SVM -> XGB        0.5236    0.8905    0.8938       0.9999           0.9097
     4598 LGBM -> KNN -> RF -> DT -> SVM -> MNB -> LR -> XGB        0.5304    0.8905    0.8938       0.9999           0.9097
     1902 LGBM -> KNN -> MNB -> DT -> SVM -> LR -> RF -> XGB        0.5399    0.8905    0.8938       0.9999           0.9097
      456 RF -> LGBM -> KNN -> DT -> LR -> SVM -> MNB -> XGB        0.5864    0.8905    0.8938       

C:\conda\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
